# Soppelrom-3D: tren kassedetektoren på GPU

**Før du starter:** Runtime → Change runtime type → **T4 GPU**

**Lokalt først** (lager `yolo_dataset.zip` du laster opp i steg 2):
```powershell
cd C:\Users\REG564367\soppelrom-3d
Compress-Archive -Path data\yolo_dataset -DestinationPath yolo_dataset.zip -Force
```

**Etterpå:** legg den nedlastede `bins_latest.pt` i `soppelrom-3d\models\` — da bruker
annoteringsverktøyet og pipeline den automatisk, og den kan pushes til GitHub.

In [ ]:
!pip -q install ultralytics
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'INGEN — bytt runtime til T4!')

In [ ]:
from google.colab import files

uploaded = files.upload()  # velg yolo_dataset.zip
!rm -rf /content/yolo_dataset && unzip -q yolo_dataset.zip -d /content/

from pathlib import Path
yaml_path = Path('/content/yolo_dataset/dataset.yaml')
lines = yaml_path.read_text(encoding='utf-8').splitlines()
lines = ['path: /content/yolo_dataset' if l.startswith('path:') else l for l in lines]
yaml_path.write_text('\n'.join(lines), encoding='utf-8')
print(yaml_path.read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data='/content/yolo_dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='bins',
)

In [ ]:
from pathlib import Path
import shutil

best = Path(results.save_dir) / 'weights' / 'best.pt'
shutil.copy2(best, '/content/bins_latest.pt')

metrics = YOLO('/content/bins_latest.pt').val(data='/content/yolo_dataset/dataset.yaml')
print('mAP50:', round(float(metrics.box.map50), 3), ' mAP50-95:', round(float(metrics.box.map), 3))

from google.colab import files
files.download('/content/bins_latest.pt')